In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor as KNN
from xgboost import XGBRegressor
from sklearn.metrics import r2_score

from sklearn.model_selection import cross_val_score
import optuna as opt


c:\Users\ACER\miniconda3\envs\agentic\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
dataset=pd.read_csv(r"D:\Downloads\agentic_reality.csv")
dataset.head(n=10)

,Project_ID,Industry,Agent_Type,Primary_Tool,Status,Failure_Reason,Human_in_the_Loop_Ratio,Tokens_per_Task,ROI_Score
0,AGP-0001,Insurance,Autonomous,LangGraph,Production,NaN,0.389,2094,2.51
1,AGP-0002,Retail,Assistant,CrewAI,Production,NaN,0.990,5993,3.78
2,AGP-0003,Energy,Multi-agent Swarm,AutoGPT 3.0,Pilot,Loop Error,0.475,9032,0.91
3,AGP-0004,Technology,Multi-agent Swarm,LangGraph,Pilot,Loop Error,0.372,8735,0.75
4,AGP-0005,Finance,Autonomous,LangGraph,Production,NaN,0.613,1645,2.58
5,AGP-0006,Telecom,Autonomous,CrewAI,Abandoned,Schema Drift,0.168,3713,-0.38
6,AGP-0007,Insurance,Multi-agent Swarm,AutoGPT 3.0,Abandoned,Schema Drift,0.478,10530,-1.51
7,AGP-0008,Insurance,Multi-agent Swarm,CrewAI,Abandoned,Cost Overrun,0.224,6478,-0.71
8,AGP-0009,Finance,Assistant,CrewAI,Production,NaN,0.795,2821,1.22
9,AGP-0010,Retail,Assistant,LangGraph,Production,NaN,0.724,4428,4.49


In [5]:
dataset=dataset.drop(columns=["Project_ID"], axis=1)
dataset

,Industry,Agent_Type,Primary_Tool,Status,Failure_Reason,Human_in_the_Loop_Ratio,Tokens_per_Task,ROI_Score
0,Insurance,Autonomous,LangGraph,Production,NaN,0.389,2094,2.51
1,Retail,Assistant,CrewAI,Production,NaN,0.990,5993,3.78
2,Energy,Multi-agent Swarm,AutoGPT 3.0,Pilot,Loop Error,0.475,9032,0.91
3,Technology,Multi-agent Swarm,LangGraph,Pilot,Loop Error,0.372,8735,0.75
4,Finance,Autonomous,LangGraph,Production,NaN,0.613,1645,2.58
...,...,...,...,...,...,...,...,...
2495,Manufacturing,Autonomous,CrewAI,Pilot,Loop Error,0.266,5788,-0.14
2496,Manufacturing,Autonomous,LangGraph,Pilot,Schema Drift,0.162,3609,1.73
2497,Healthcare,Assistant,CrewAI,Pilot,Security Violation,0.799,3182,0.63
2498,Energy,Assistant,CrewAI,Pilot,Loop Error,0.571,5360,-0.28


In [15]:
dataset['Industry'].name

'Industry'

In [13]:
print(dataset['Failure_Reason'].value_counts())
print('-----------------------------')
print(dataset['Agent_Type'].value_counts())
print('-----------------------------')
print(dataset['Primary_Tool'].value_counts())
print('-----------------------------')
print(dataset['Status'].value_counts())

Failure_Reason
Loop Error            713
Cost Overrun          576
Schema Drift          439
Security Violation    426
No Failure            346
Name: count, dtype: int64
-----------------------------
Agent_Type
Assistant            988
Autonomous           918
Multi-agent Swarm    594
Name: count, dtype: int64
-----------------------------
Primary_Tool
LangGraph      1059
CrewAI          851
AutoGPT 3.0     590
Name: count, dtype: int64
-----------------------------
Status
Pilot         1267
Abandoned      887
Production     346
Name: count, dtype: int64


In [14]:
print(dataset['Industry'].value_counts())

Industry
Finance          410
Technology       345
Healthcare       329
Retail           318
Manufacturing    217
Government       199
Insurance        190
Legal            184
Energy           155
Telecom          153
Name: count, dtype: int64


In [6]:
dataset["Failure_Reason"].fillna("No Failure", inplace=True)

In [7]:
dataset.isnull().sum()

Industry                   0
Agent_Type                 0
Primary_Tool               0
Status                     0
Failure_Reason             0
Human_in_the_Loop_Ratio    0
Tokens_per_Task            0
ROI_Score                  0
dtype: int64

In [86]:
unique_indus=dataset["Industry"].unique()
print(unique_indus)
unique_Agent=dataset["Agent_Type"].unique()
print(unique_Agent)
unique_Tool=dataset["Primary_Tool"].unique()
print(unique_Tool)
unique_Status=dataset["Status"].unique()
print(unique_Status)
unique_Failure=dataset["Failure_Reason"].unique()
print(unique_Failure)


['Insurance' 'Retail' 'Energy' 'Technology' 'Finance' 'Telecom'
 'Government' 'Healthcare' 'Manufacturing' 'Legal']
['Autonomous' 'Assistant' 'Multi-agent Swarm']
['LangGraph' 'CrewAI' 'AutoGPT 3.0']
['Production' 'Pilot' 'Abandoned']
['No Failure' 'Loop Error' 'Schema Drift' 'Cost Overrun'
 'Security Violation']


In [87]:
dataset.describe()

,Human_in_the_Loop_Ratio,Tokens_per_Task,ROI_Score
count,2500.000000,2500.000000,2500.000000
mean,0.484933,5148.214800,0.425728
std,0.271753,2473.059822,1.272453
min,0.010000,500.000000,-2.000000
25%,0.239750,3491.000000,-0.470000
50%,0.483000,4655.500000,0.160000
75%,0.724000,6322.500000,1.060000
max,0.990000,20155.000000,5.000000


In [88]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Industry                 2500 non-null   object 
 1   Agent_Type               2500 non-null   object 
 2   Primary_Tool             2500 non-null   object 
 3   Status                   2500 non-null   object 
 4   Failure_Reason           2500 non-null   object 
 5   Human_in_the_Loop_Ratio  2500 non-null   float64
 6   Tokens_per_Task          2500 non-null   int64  
 7   ROI_Score                2500 non-null   float64
dtypes: float64(2), int64(1), object(5)
memory usage: 156.4+ KB


In [89]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Industry                 2500 non-null   object 
 1   Agent_Type               2500 non-null   object 
 2   Primary_Tool             2500 non-null   object 
 3   Status                   2500 non-null   object 
 4   Failure_Reason           2500 non-null   object 
 5   Human_in_the_Loop_Ratio  2500 non-null   float64
 6   Tokens_per_Task          2500 non-null   int64  
 7   ROI_Score                2500 non-null   float64
dtypes: float64(2), int64(1), object(5)
memory usage: 156.4+ KB


In [16]:
dataset

,Industry,Agent_Type,Primary_Tool,Status,Failure_Reason,Human_in_the_Loop_Ratio,Tokens_per_Task,ROI_Score
0,Insurance,Autonomous,LangGraph,Production,No Failure,0.389,2094,2.51
1,Retail,Assistant,CrewAI,Production,No Failure,0.990,5993,3.78
2,Energy,Multi-agent Swarm,AutoGPT 3.0,Pilot,Loop Error,0.475,9032,0.91
3,Technology,Multi-agent Swarm,LangGraph,Pilot,Loop Error,0.372,8735,0.75
4,Finance,Autonomous,LangGraph,Production,No Failure,0.613,1645,2.58
...,...,...,...,...,...,...,...,...
2495,Manufacturing,Autonomous,CrewAI,Pilot,Loop Error,0.266,5788,-0.14
2496,Manufacturing,Autonomous,LangGraph,Pilot,Schema Drift,0.162,3609,1.73
2497,Healthcare,Assistant,CrewAI,Pilot,Security Violation,0.799,3182,0.63
2498,Energy,Assistant,CrewAI,Pilot,Loop Error,0.571,5360,-0.28


We can do the label encoding by a new function of the pandas "get_dummies". Just pass all the categorical columns and it will convert each by label encoding.

In [91]:
# dataset = pd.get_dummies(dataset, columns=["Industry", "Agent_Type", "Primary_Tool", "Status", "Failure_Reason"], drop_first=True)

In [19]:
encoder = OneHotEncoder(sparse_output=False)

one_hot_encoded = encoder.fit_transform(dataset[dataset.columns[dataset.dtypes == "object"]])

one_hot_df = pd.DataFrame(one_hot_encoded, 
                          columns=encoder.get_feature_names_out(dataset.columns[dataset.dtypes == "object"]))

dataset= pd.concat([dataset.drop(dataset.columns[dataset.dtypes == "object"], axis=1), one_hot_df], axis=1)

In [20]:
dataset

,Human_in_the_Loop_Ratio,Tokens_per_Task,ROI_Score,Industry_Energy,Industry_Finance,Industry_Government,Industry_Healthcare,Industry_Insurance,Industry_Legal,Industry_Manufacturing,...,Primary_Tool_CrewAI,Primary_Tool_LangGraph,Status_Abandoned,Status_Pilot,Status_Production,Failure_Reason_Cost Overrun,Failure_Reason_Loop Error,Failure_Reason_No Failure,Failure_Reason_Schema Drift,Failure_Reason_Security Violation
0,0.389,2094,2.51,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
1,0.990,5993,3.78,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
2,0.475,9032,0.91,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
3,0.372,8735,0.75,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
4,0.613,1645,2.58,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2495,0.266,5788,-0.14,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
2496,0.162,3609,1.73,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2497,0.799,3182,0.63,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
2498,0.571,5360,-0.28,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0


In [94]:
# dataset["Industry"] = dataset["Industry"].map({ key:index for index,key in enumerate(unique_indus)})
# dataset["Agent_Type"] = dataset["Agent_Type"].map({ key:index for index,key in enumerate(unique_Agent)})
# dataset["Primary_Tool"] = dataset["Primary_Tool"].map({ key:index for index,key in enumerate(unique_Tool)})
# dataset["Status"] = dataset["Status"].map({ key:index for index,key in enumerate(unique_Status)})
# dataset["Failure_Reason"] = dataset["Failure_Reason"].map({ key:index for index,key in enumerate(unique_Failure)})

In [95]:
dataset

,Human_in_the_Loop_Ratio,Tokens_per_Task,ROI_Score,Industry_Energy,Industry_Finance,Industry_Government,Industry_Healthcare,Industry_Insurance,Industry_Legal,Industry_Manufacturing,...,Primary_Tool_CrewAI,Primary_Tool_LangGraph,Status_Abandoned,Status_Pilot,Status_Production,Failure_Reason_Cost Overrun,Failure_Reason_Loop Error,Failure_Reason_No Failure,Failure_Reason_Schema Drift,Failure_Reason_Security Violation
0,0.389,2094,2.51,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
1,0.990,5993,3.78,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
2,0.475,9032,0.91,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
3,0.372,8735,0.75,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
4,0.613,1645,2.58,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2495,0.266,5788,-0.14,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
2496,0.162,3609,1.73,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2497,0.799,3182,0.63,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
2498,0.571,5360,-0.28,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0


In [96]:
x_train, x_test, y_train, y_test = train_test_split(dataset.drop(columns=["ROI_Score"], axis=1), dataset["ROI_Score"], test_size=0.2, random_state=42)
x_train.to_csv(r"D:\End to end project\Learning_MLOps\data_save\x_train.csv", index=False)
x_test.to_csv(r"D:\End to end project\Learning_MLOps\data_save\x_test.csv", index=False)
y_train.to_csv(r"D:\End to end project\Learning_MLOps\data_save\y_train.csv", index=False)
y_test.to_csv(r"D:\End to end project\Learning_MLOps\data_save\y_test.csv", index=False)

In [97]:
scaler=StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)

# Here we use a new library named Optuna which is use for finding the best Model and its best Hyperparameters. We Have to define a objective funtion and in that function we declare all our candidate models and there repective hyperparameters.

In [98]:
def objective0(trial):
    regressor=trial.suggest_categorical("regressor", ["RandomForestRegressor", "XGBRegressor"])
    
    if regressor=="RandomForestRegressor":
        n_estimators=trial.suggest_int("n_estimators", 10, 200)
        max_depth=trial.suggest_int("max_depth", 5, 10)
        min_samples_split=trial.suggest_int("min_samples_split", 2, 10)
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 10)
        
        model=RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, random_state=42, min_samples_split=min_samples_split, min_samples_leaf=min_samples_leaf)
    elif regressor=="XGBRegressor":
        n_estimators=trial.suggest_int("n_estimators", 10, 200)
        max_depth=trial.suggest_int("max_depth", 2, 10)
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.03)
        
        model=XGBRegressor(n_estimators=n_estimators, max_depth=max_depth, learning_rate=learning_rate, random_state=42)

    

    score=cross_val_score(model, x_train, y_train, cv=5, scoring="r2").mean()
    return score
        

In [99]:
def objective1(trial):
    regressor=trial.suggest_categorical("regressor", ["ElasticNet", "SVR", "KNN"])
    if regressor=="ElasticNet":
        alpha=trial.suggest_float("alpha", 0.001, 10.0)
        l1_ratio=trial.suggest_float("l1_ratio", 0.0, 1.0)
        
        model=ElasticNet(alpha=alpha, l1_ratio=l1_ratio, random_state=42)
    elif regressor=="SVR":
        C=trial.suggest_float("C", 0.1, 10.0)
        gamma=trial.suggest_float("gamma", 0.001, 0.01)
        
        model=SVR(C=C, gamma=gamma)
    elif regressor=="KNN":
        n_neighbors=trial.suggest_int("n_neighbors", 1, 30)
        model=KNN(n_neighbors=n_neighbors)

    
    score=cross_val_score(model, x_train_scaled, y_train, cv=5, scoring="r2").mean()
    return score
        

In [100]:
study0=opt.create_study(direction="maximize")
study0.optimize(objective0, n_trials=100)


[I 2026-04-29 15:20:06,319] A new study created in memory with name: no-name-89896b26-1577-4e03-a294-01a1e8c4bd44


[I 2026-04-29 15:20:06,666] Trial 0 finished with value: 0.3681297046639284 and parameters: {'regressor': 'XGBRegressor', 'n_estimators': 21, 'max_depth': 10, 'learning_rate': 0.017960692876459242}. Best is trial 0 with value: 0.3681297046639284.
[I 2026-04-29 15:20:06,977] Trial 1 finished with value: 0.7087064142944282 and parameters: {'regressor': 'RandomForestRegressor', 'n_estimators': 20, 'max_depth': 5, 'min_samples_split': 4, 'min_samples_leaf': 4}. Best is trial 1 with value: 0.7087064142944282.
[I 2026-04-29 15:20:07,190] Trial 2 finished with value: 0.32457866826021686 and parameters: {'regressor': 'XGBRegressor', 'n_estimators': 25, 'max_depth': 7, 'learning_rate': 0.012378730282475839}. Best is trial 1 with value: 0.7087064142944282.
[I 2026-04-29 15:20:07,717] Trial 3 finished with value: 0.6966804824803281 and parameters: {'regressor': 'RandomForestRegressor', 'n_estimators': 17, 'max_depth': 9, 'min_samples_split': 4, 'min_samples_leaf': 4}. Best is trial 1 with value: 

In [101]:
print(study0.best_trial.params)
print(study0.best_trial.value)

{'regressor': 'XGBRegressor', 'n_estimators': 187, 'max_depth': 2, 'learning_rate': 0.022294413883118733}
0.711617456354529


In [102]:
study1=opt.create_study(direction="maximize")
study1.optimize(objective1, n_trials=100)

[I 2026-04-29 15:21:25,367] A new study created in memory with name: no-name-86eef39b-6b70-4707-89cb-aec0a64d5876
[I 2026-04-29 15:21:25,408] Trial 0 finished with value: 0.6718024371512478 and parameters: {'regressor': 'ElasticNet', 'alpha': 0.18849029935014133, 'l1_ratio': 0.6788200393744057}. Best is trial 0 with value: 0.6718024371512478.
[I 2026-04-29 15:21:25,436] Trial 1 finished with value: 0.39374487248613355 and parameters: {'regressor': 'ElasticNet', 'alpha': 1.2321595043609774, 'l1_ratio': 0.35126096241239435}. Best is trial 0 with value: 0.6718024371512478.
[I 2026-04-29 15:21:25,639] Trial 2 finished with value: 0.6313608478349576 and parameters: {'regressor': 'KNN', 'n_neighbors': 11}. Best is trial 0 with value: 0.6718024371512478.
[I 2026-04-29 15:21:26,717] Trial 3 finished with value: 0.6747486688365607 and parameters: {'regressor': 'SVR', 'C': 9.730047313501338, 'gamma': 0.009296052994117038}. Best is trial 3 with value: 0.6747486688365607.
[I 2026-04-29 15:21:27,70

In [103]:
print(study1.best_trial.params)
print(study1.best_trial.value)

{'regressor': 'SVR', 'C': 7.015283273654784, 'gamma': 0.0010137226360850196}
0.6989907491470608


In [104]:
best_model = XGBRegressor(n_estimators=study0.best_trial.params["n_estimators"], 
                    max_depth=study0.best_trial.params["max_depth"], 
                    learning_rate=study0.best_trial.params["learning_rate"], 
                    random_state=42)

best_model.fit(x_train, y_train)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_met

In [105]:
y_pred=best_model.predict(x_test)

In [106]:
accuracy= r2_score(y_test, y_pred)
print("Test Accuracy:", accuracy)

Test Accuracy: 0.7151064351736465


# At this moment we are not focusing on the Accuracy of the model, here we just implementing the full End to End pipeline of MLops 

In [107]:
import joblib
joblib.dump(best_model, r"D:\End to end project\Learning_MLOps\model_registry\best_model.pkl")

['D:\\End to end project\\Learning_MLOps\\model_registry\\best_model.pkl']